# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # This is a Metadata object, not a dict.
print(f"{meta.name}: {meta.description}")

# Optionally, show keywords and other summary info
print("\nKeywords:", getattr(meta, 'keywords', None))
print("License:", getattr(meta, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their IDs. All lookups by Croissant `@id`.

In [ ]:
# List available record sets by @id
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- ID: {record_set['@id']} | Name: {record_set['name']}")
    # List fields (columns)
    if 'fields' in record_set:
        for field in record_set['fields']:
            print(f"    Field ID: {field['@id']}, Name: {field.get('name', field['@id'])}, Type: {field.get('dataType', 'unknown')}")

## 3. Data Extraction
Load data from a specific record set (using its `@id`) into a DataFrame for analysis. All IDs are referred by Croissant `@id`.

In [ ]:
# Extract data from each record set. Collect all record set IDs.
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet: {rs_id}")

# Show columns for the first record set (replace by a specific @id if needed)
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"RecordSet ID: {first_rs_id}\nFields/Columns: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, categorizing, groupings, and removing outliers. All fields referenced by Croissant `@id`.

In [ ]:
import numpy as np

# Choose record set and a numeric field by their @id
# Replace these IDs with the actual ones from the overview section if needed.
# We'll try to work with the first non-empty frame and first numeric field detected.
main_rs_id = None
numeric_field_id = None

for rs_id, df in dataframes.items():
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            main_rs_id = rs_id
            numeric_field_id = col
            break
    if main_rs_id:
        break

if not main_rs_id or not numeric_field_id:
    raise RuntimeError("No numeric data found to analyze.")

main_df = dataframes[main_rs_id]

# Filtering
threshold = main_df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()

print(f"Filtered records on {numeric_field_id} > {threshold:.3f} for RecordSet {main_rs_id}:")
display(filtered_df.head())

# Normalization
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping on a likely categorical field (first non-numeric column)
group_field_id = None
for col in main_df.columns:
    if not np.issubdtype(main_df[col].dtype, np.number):
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (first 5):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Basic Distribution Plot
plt.figure(figsize=(8,5))
if numeric_field_id:
    main_df[numeric_field_id].hist(bins=30, color='skyblue', edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group field if present
if group_field_id:
    plt.figure(figsize=(10,5))
    filtered_df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, rot=90)
    plt.title(f"{numeric_field_id} by {group_field_id} for RecordSet {main_rs_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset's metadata and records using the Croissant schema via `mlcroissant`. We examined available record sets, performed basic filtering and normalization on numeric fields referenced by their Croissant `@id`, and visualized key field distributions. For further analysis, consult the Croissant record set structure through the schema and map all fields with their exact `@id` for programmatic data integration.

You can now extend this notebook to perform deeper analysis, modeling, or integrate with other FAIR-compliant data tools.